# EDA de las predicciones: descripciones de escenas controladas (Multimodal3DIdent, *test*)

**Pregunta.** ¿Qué dicen realmente los modelos cuando describen una imagen cuyos factores generativos conocemos, y en qué fallan?
Este notebook es exploratorio: caracteriza las salidas de cada corrida *etapa × modelo × prompt* antes (y como complemento) de las métricas
de `02. Evaluación E2`.

Cada imagen $i$ se genera a partir de
$$
z_i = \big(s_i,\; x_i,\; y_i,\; h^{\text{obj}}_i,\; h^{\text{spot}}_i,\; h^{\text{bg}}_i\big),\qquad
s_i\in\{0,\dots,6\},\;(x_i,y_i)\in\{0,1,2\}^2,\;h^{(\cdot)}_i\in[0,1),
$$
y cada corrida $r$ produce un texto $\hat t_{i,r}$ del que se extrae léxicamente una estimación $\hat z_{i,r}$ atributo por atributo.

**Contenido**

| § | Tema | Figuras |
|---|---|---|
| 1 | Configuración, carga de `runs/*.jsonl` y parseo de `exp_id` | diseño experimental |
| 2 | Integridad y factores verdaderos | distribución de factores |
| 3 | Extractores: léxicos, colores con nombre `xkcd:`/`tab:` y calibración con la referencia | validación de tono |
| 4 | Texto: longitud, truncamiento, repetición, vocabulario | longitud y truncamiento |
| 5 | Fidelidad por atributo | heatmaps, IC de Wilson, descomposición del error |
| 6 | Sensibilidad al prompt, cobertura vs. exactitud, similitud textual vs. fidelidad, etapas | 4 figuras |
| 7 | Errores con el mejor prompt de cada modelo | confusión, grilla de posición, tono, contraste |
| 8 | Comparaciones pareadas (McNemar exacto + Holm) | Δ entre modelos |
| 9 | Dificultad por imagen y ejemplos | dificultad |
| 10 | Exportación y limitaciones | — |

> **VL‑JEPA** no genera texto: recupera un caption de un *pool*. Sus predicciones tienen el formato de la referencia,
> por lo que su similitud textual está inflada respecto de los modelos generativos, mientras que la fidelidad por atributo sí es comparable.

## 1. Configuración

In [ ]:
import re
import sys
import colorsys
import itertools
import time
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy import stats
from IPython.display import display, Markdown

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RUNS_DIR = ROOT / 'runs'
MANIFEST = ROOT / 'data/manifests/m3di_test.parquet'
FIG_DIR = ROOT / "reports/figures/eval"
TAB_DIR = ROOT / "reports/tables/eval"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

# ── Parámetros ───────────────────────────────────────────────────────────────
SEED = 20260914
ALPHA = 0.05
KEEP = {'dataset': 'm3di', 'split': 'test'}          # filtro sobre lo parseado de exp_id (None = no filtrar)
FIELD_ALIASES = {
    'image_id':   ['image_id', 'id', 'idx', 'index'],
    'prediction': ['prediction', 'pred', 'output', 'text', 'caption_pred', 'generated'],
    'reference':  ['caption_ref', 'caption', 'reference', 'caption_gt', 'ref'],
}
RETRIEVAL_MODELS = {'vljepa', 'vl-jepa', 'vl_jepa'}   # recuperan del pool, no generan
# Nombre legible por prefijo del token de modelo en exp_id (el resto del token se agrega entre paréntesis)
MODEL_PREFIX_LABELS = [('llava', 'LLaVA-1.5-7B'), ('qwen', 'Qwen2.5-VL-7B'),
                       ('internvl', 'InternVL3.5-8B'), ('vljepa', 'VL-JEPA'), ('vl_jepa', 'VL-JEPA'),
                       ('vl-jepa', 'VL-JEPA')]
PROMPT_LABELS = {}                                     # opcional, p. ej. {'p0_minimal': 'P0 mínimo'}
N_EXAMPLES = 3

RNG = np.random.default_rng(SEED)
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 220)
pd.set_option('display.max_colwidth', 150)
warnings.filterwarnings('ignore', category=FutureWarning)
T0 = time.time()

In [ ]:
# ── Estilo de figuras ────────────────────────────────────────────────────────
SERIES = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300', '#4a3aa7', '#e34948']
INK, INK2, INK3, GRID, SURF, NEUTRAL = '#0b0b0b', '#52514e', '#8a8984', '#e6e5e1', '#fcfcfb', '#c9c8c2'
SEQ = mcolors.LinearSegmentedColormap.from_list(
    'seq_blue', ['#f4f8fe', '#cde2fb', '#9ec5f4', '#6da7ec', '#3987e5', '#256abf', '#184f95', '#0d366b'])
DIV = mcolors.LinearSegmentedColormap.from_list(
    'div_red_blue', ['#c23a39', '#ee9190', '#f0efec', '#6da7ec', '#184f95'])   # azul = mejor
MARKERS = ['o', 's', '^', 'D', 'v', 'P', 'X', '*']

plt.rcParams.update({
    'figure.facecolor': SURF, 'axes.facecolor': SURF, 'savefig.facecolor': SURF,
    'axes.edgecolor': INK3, 'axes.labelcolor': INK2, 'axes.titlecolor': INK,
    'axes.titlesize': 10.5, 'axes.titleweight': 'semibold', 'axes.labelsize': 9.5,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.8, 'axes.axisbelow': True,
    'xtick.color': INK2, 'ytick.color': INK2, 'xtick.labelsize': 8.5, 'ytick.labelsize': 8.5,
    'legend.frameon': False, 'legend.fontsize': 8.5, 'font.size': 9.5,
    'lines.linewidth': 2, 'lines.markersize': 6, 'figure.dpi': 110,
})
PCT = plt.FuncFormatter(lambda v, _: f'{v:.0%}')


def savefig(fig, name):
    for ext in ('png', 'pdf'):
        fig.savefig(FIG_DIR / f'{name}.{ext}', dpi=200, bbox_inches='tight')


def save_table(df, name, index=True, float_format='%.3f'):
    df.to_csv(TAB_DIR / f'{name}.csv', index=index)
    try:
        df.to_latex(TAB_DIR / f'{name}.tex', index=index, float_format=float_format, escape=True, na_rep='--')
    except Exception as e:
        print(f'[aviso] {name}.tex no generado: {e}')


def heat(ax, M, xlabels, ylabels, cmap=SEQ, vmin=0, vmax=1, fmt='{:.2f}', thresh=0.55, fontsize=7.5):
    # Heatmap con celdas separadas, valores anotados y tinta que contrasta con el relleno.
    M = np.asarray(M, float)
    ax.pcolormesh(M, cmap=cmap, vmin=vmin, vmax=vmax, edgecolors=SURF, linewidth=1.5)
    for (r, c), v in np.ndenumerate(M):
        if not np.isnan(v):
            rel = (v - vmin) / (vmax - vmin + 1e-12)
            dark = rel > thresh if cmap is SEQ else abs(rel - 0.5) > 0.3
            ax.text(c + .5, r + .5, fmt.format(v), ha='center', va='center', fontsize=fontsize,
                    color='white' if dark else INK)
    ax.set_xticks(np.arange(len(xlabels)) + .5, xlabels)
    ax.set_yticks(np.arange(len(ylabels)) + .5, ylabels)
    ax.invert_yaxis(); ax.grid(False); ax.tick_params(length=0)
    for s in ax.spines.values():
        s.set_visible(False)


def plain_log_x(ax):
    # Eje x logarítmico con marcas legibles (5, 10, 20, 50…) en vez de notación 10^k.
    lo, hi = ax.get_xlim()
    ticks = [t for t in (1, 2, 5, 10, 20, 50, 100, 200, 500, 1000, 2000) if lo <= t <= hi]
    ax.set_xticks(ticks, [f'{t:g}' for t in ticks])
    ax.xaxis.set_minor_formatter(plt.NullFormatter())


def to_pandas(df: pl.DataFrame) -> pd.DataFrame:
    # Conversión sin depender de pyarrow.
    return pd.DataFrame(df.to_dict(as_series=False))

### 1.1 Parseo de `exp_id`

Formato: `e{k}_{dataset}_{split}_{modelo}_p{j}[_{nombre}][_n{N}[_{sufijo}]]`, p. ej. `e1_m3di_test_internvl_p0_minimal_n1000`.
El modelo se captura de forma **no codiciosa** hasta el primer `_p{j}` (admite `qwen2_5_vl`); si la regex falla, un respaldo por tokens
ubica el *split* conocido y el primer `p{j}` posterior. `exp` identifica la etapa.

In [ ]:
EXP_COLS = ['exp', 'dataset', 'split', 'model', 'prompt_id', 'prompt_name', 'n_samples', 'suffix']
META_COLS = EXP_COLS[:6] + ['prompt', 'n_samples', 'suffix']
SPLITS = {'train', 'val', 'valid', 'test'}
EXP_RE = re.compile(
    r'^(?P<exp>e\d+)_(?P<dataset>[^_]+)_(?P<split>[^_]+)_'
    r'(?P<model>.+?)_(?P<prompt_id>p\d+)(?:_(?P<prompt_name>.+?))?'
    r'(?:_n(?P<n_samples>\d+)(?:_(?P<suffix>.+))?)?$', re.I)


def _parse_tokens(exp_id):
    tok = exp_id.split('_')
    i_split = next((i for i, t in enumerate(tok) if t.lower() in SPLITS), None)
    i_p = next((i for i, t in enumerate(tok)
                if re.fullmatch(r'p\d+', t, re.I) and (i_split is None or i > i_split + 1)), None)
    if i_split is None or i_p is None:
        return None
    rest = tok[i_p + 1:]
    i_n = next((i for i, t in enumerate(rest) if re.fullmatch(r'n\d+', t)), None)
    return {'exp': '_'.join(tok[:max(i_split - 1, 0)]) or None,
            'dataset': tok[i_split - 1] if i_split > 0 else None, 'split': tok[i_split],
            'model': '_'.join(tok[i_split + 1:i_p]), 'prompt_id': tok[i_p],
            'prompt_name': '_'.join(rest[:i_n] if i_n is not None else rest) or None,
            'n_samples': rest[i_n][1:] if i_n is not None else None,
            'suffix': ('_'.join(rest[i_n + 1:]) or None) if i_n is not None else None}


def parse_exp_id(exp_id) -> dict:
    s = str(exp_id).strip()
    m = EXP_RE.match(s)
    d = m.groupdict() if m else _parse_tokens(s)
    if d is None:
        return {'exp_id': exp_id, 'parse_ok': False, **dict.fromkeys(META_COLS)}
    d = {k: d.get(k) for k in EXP_COLS}
    for k in ('exp', 'dataset', 'split', 'prompt_id'):
        d[k] = d[k].lower() if d[k] else d[k]
    d['n_samples'] = int(d['n_samples']) if d['n_samples'] else None
    d['prompt'] = f"{d['prompt_id']}_{d['prompt_name']}" if d['prompt_name'] else d['prompt_id']
    return {'exp_id': exp_id, 'parse_ok': True, **d}


def model_label(token: str) -> tuple[str, str]:
    # Devuelve (etiqueta, modelo base) a partir del token de exp_id.
    t = token.lower()
    for pref, lab in MODEL_PREFIX_LABELS:
        if t.startswith(pref):
            rest = re.sub(r'^[_\-]+', '', token[len(pref):])
            rest = re.sub(r'^(?:2[_.]?5[_\-]?vl|3[_.]?5|1[_.]?5)[_\-]?', '', rest, flags=re.I)  # versión ya incluida
            return (f'{lab} ({rest})' if rest else lab), lab
    return token, token

### 1.2 Carga

Se leen `runs/*.jsonl` (el `exp_id` es el nombre del archivo, como en `02. Evaluación E2`); si no hay, se buscan `.jsonl`/`.parquet`
recursivamente. Los nombres de campo se resuelven con `FIELD_ALIASES`; atributos y referencia faltantes se completan desde el manifest.
Las columnas `model`/`prompt` que traiga el archivo se ignoran: provienen del *split* incorrecto del identificador.

In [ ]:
def resolve(cols, aliases):
    lower = {c.lower(): c for c in cols}
    return next((lower[a.lower()] for a in aliases if a.lower() in lower), None)


def canon_id(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip()
    return s.str.lstrip('0').replace('', '0') if s.str.fullmatch(r'\d+').all() else s


ATTR_COLS = ['object_shape', 'object_xpos', 'object_ypos', 'object_color', 'spotlight_color', 'background_color']
manifest = to_pandas(pl.read_parquet(MANIFEST))
m_id = resolve(manifest.columns, FIELD_ALIASES['image_id'])
manifest = manifest.rename(columns={m_id: 'image_id'})
manifest['image_id'] = manifest['image_id'].astype(str)
manifest['_key'] = canon_id(manifest['image_id'])
assert manifest['_key'].is_unique, 'image_id duplicado en el manifest'
assert not (miss := [c for c in ATTR_COLS if c not in manifest.columns]), f'Faltan en el manifest: {miss}'
m_ref = resolve(manifest.columns, FIELD_ALIASES['reference'])
for c in ['object_color', 'spotlight_color', 'background_color']:
    if manifest[c].min() < 0 or manifest[c].max() > 1:
        print(f'[aviso] {c} fuera de [0,1]; se asume tono y se aplica módulo 1.')
        manifest[c] = np.mod(manifest[c], 1.0)
print(f'Manifest: {len(manifest):,} imágenes')

files = sorted(RUNS_DIR.glob('*.jsonl')) or sorted(
    p for ext in ('*.jsonl', '*.parquet') for p in RUNS_DIR.rglob(ext) if not p.name.endswith('.pool.parquet'))
frames, skipped, metas = [], [], []
for f in files:
    exp_id = f.stem if f.parent == RUNS_DIR else f.parent.name
    meta = parse_exp_id(exp_id)
    if not meta['parse_ok']:
        skipped.append((f.name, 'exp_id sin formato reconocible')); continue
    if KEEP and any(v is not None and meta[k] != v.lower() for k, v in KEEP.items()):
        skipped.append((f.name, f'excluido por KEEP={KEEP}')); continue
    try:
        d = pl.read_ndjson(f, infer_schema_length=None) if f.suffix == '.jsonl' else pl.read_parquet(f)
    except Exception as e:
        skipped.append((f.name, f'error de lectura: {e}')); continue
    c_id, c_pred = resolve(d.columns, FIELD_ALIASES['image_id']), resolve(d.columns, FIELD_ALIASES['prediction'])
    c_ref = resolve(d.columns, FIELD_ALIASES['reference'])
    if c_pred is None:
        skipped.append((f.name, 'sin columna de predicción')); continue
    if 'exp_id' in d.columns and d['exp_id'].n_unique() == 1 and d['exp_id'][0] != exp_id:
        meta = parse_exp_id(d['exp_id'][0]) if parse_exp_id(d['exp_id'][0])['parse_ok'] else meta
    sel = [pl.col(c_pred).cast(pl.Utf8).fill_null('').alias('prediction')]
    sel.append((pl.col(c_id).cast(pl.Utf8) if c_id else pl.int_range(pl.len()).cast(pl.Utf8)).alias('image_id'))
    if c_ref:
        sel.append(pl.col(c_ref).cast(pl.Utf8).alias('reference'))
    frames.append(d.select(sel).with_columns(pl.lit(meta['exp_id']).alias('exp_id')))
    metas.append(meta)

assert frames, f'No se cargó ninguna corrida desde {RUNS_DIR}'
raw = to_pandas(pl.concat(frames, how='diagonal_relaxed'))
runs_meta = pd.DataFrame(metas).drop(columns='parse_ok').drop_duplicates('exp_id')
print(f'{len(frames)} corridas · {len(raw):,} filas')
if skipped:
    display(pd.DataFrame(skipped, columns=['archivo', 'motivo']))

In [ ]:
runs_meta[['model_label', 'model_base']] = runs_meta['model'].map(model_label).apply(pd.Series)
runs_meta['prompt_label'] = runs_meta['prompt'].map(lambda p: PROMPT_LABELS.get(p, p))
runs_meta['retrieval'] = runs_meta['model'].str.lower().isin(RETRIEVAL_MODELS) | \
                         runs_meta['model_base'].eq('VL-JEPA')
MULTI_EXP = runs_meta['exp'].nunique() > 1
runs_meta['run'] = ((runs_meta['exp'] + ' · ') if MULTI_EXP else '') + runs_meta['model_label'] + ' · ' + runs_meta['prompt_label']

KNOWN = [lab for _, lab in MODEL_PREFIX_LABELS]
BASE_ORDER = [m for m in dict.fromkeys(KNOWN) if m in set(runs_meta['model_base'])] + \
             sorted(set(runs_meta['model_base']) - set(KNOWN))
EXP_ORDER = sorted(runs_meta['exp'].unique(), key=lambda e: int(re.sub(r'\D', '', e) or 0))
PROMPT_ORDER = (runs_meta[['prompt_label', 'prompt_id']].drop_duplicates()
                .assign(_k=lambda t: t['prompt_id'].str[1:].astype(int)).sort_values(['_k', 'prompt_label'])
                ['prompt_label'].tolist())
runs_meta = (runs_meta.assign(_e=runs_meta['exp'].map(EXP_ORDER.index), _b=runs_meta['model_base'].map(BASE_ORDER.index),
                              _p=runs_meta['prompt_label'].map(PROMPT_ORDER.index))
             .sort_values(['_e', '_b', 'model_label', '_p']).drop(columns=['_e', '_b', '_p']).reset_index(drop=True))
RUN_ORDER = runs_meta['run'].tolist()
MODEL_ORDER = list(dict.fromkeys(runs_meta['model_label']))
BASE_COLORS = dict(zip(BASE_ORDER, SERIES))
MODEL_COLORS = {m: BASE_COLORS[b] for m, b in runs_meta[['model_label', 'model_base']].drop_duplicates().values}
RUN_COLORS = dict(zip(runs_meta['run'], runs_meta['model_label'].map(MODEL_COLORS)))
EXP_MARKERS = dict(zip(EXP_ORDER, MARKERS))

dup = runs_meta.groupby(['exp', 'model_label', 'prompt_label']).size()
assert (dup == 1).all(), f'Corridas duplicadas para la misma combinación:\n{dup[dup > 1]}'

df = raw.merge(runs_meta, on='exp_id', how='inner')
display(runs_meta[['exp_id', 'exp', 'model', 'model_label', 'prompt', 'n_samples', 'retrieval', 'run']])

### 1.3 Diseño experimental

La grilla *modelo × prompt* no tiene por qué estar completa (23 corridas). La figura muestra qué combinaciones existen por etapa y con cuántas
predicciones; las comparaciones entre prompts sólo son pareadas dentro de las celdas presentes.

In [ ]:
cnt = df.groupby(['exp', 'model_label', 'prompt_label']).size()
fig, axes = plt.subplots(1, len(EXP_ORDER), figsize=(1.0 * len(PROMPT_ORDER) * len(EXP_ORDER) + 3.5, 0.5 * len(MODEL_ORDER) + 1.6),
                         squeeze=False)
nmax = cnt.max()
for ax, e in zip(axes[0], EXP_ORDER):
    M = np.full((len(MODEL_ORDER), len(PROMPT_ORDER)), np.nan)
    for (ee, m, p), n in cnt.items():
        if ee == e:
            M[MODEL_ORDER.index(m), PROMPT_ORDER.index(p)] = n
    heat(ax, M, PROMPT_ORDER, MODEL_ORDER, fmt='{:,.0f}', vmin=0, vmax=nmax * 1.6)
    ax.set_title(f'Etapa {e}' if MULTI_EXP else 'Corridas disponibles')
    ax.tick_params(axis='x', rotation=30)
    for lab in ax.get_xticklabels():
        lab.set_ha('right')
fig.suptitle('Diseño experimental (n predicciones por corrida)', x=0.01, ha='left', fontweight='semibold')
fig.tight_layout()
savefig(fig, '00_diseno_experimental')
plt.show()

## 2. Integridad y factores verdaderos

Por corrida: filas, duplicados `(exp_id, image_id)`, imágenes fuera del manifest, imágenes sin predicción y predicciones vacías.
Los atributos verdaderos se toman **siempre del manifest**.

In [ ]:
df['_key'] = canon_id(df['image_id'])
keys = set(manifest['_key'])
integ = (df.groupby('run', sort=False)
           .agg(filas=('_key', 'size'), duplicados=('_key', lambda s: int(s.duplicated().sum())),
                fuera_manifest=('_key', lambda s: int((~s.isin(keys)).sum())),
                vacias=('prediction', lambda s: int(s.str.strip().eq('').sum())))
           .reindex(RUN_ORDER))
integ['sin_prediccion'] = [len(keys - set(df.loc[df['run'] == r, '_key'])) for r in integ.index]
integ['cobertura'] = 1 - integ['sin_prediccion'] / len(keys)
display(integ.style.format({'cobertura': '{:.1%}'}))
save_table(integ, 'integridad_corridas')

df = df.drop_duplicates(['exp_id', '_key'], keep='last')
df = df[df['_key'].isin(keys)]
keep_man = ['_key'] + ATTR_COLS + ([m_ref] if m_ref and m_ref != 'reference' else [])
df = df.merge(manifest[keep_man], on='_key', how='left')
if 'reference' not in df.columns or df['reference'].isna().all():
    assert m_ref, 'No hay referencia ni en las predicciones ni en el manifest'
    df['reference'] = df[m_ref]
df['reference'] = df['reference'].fillna('').astype(str)
df = df.reset_index(drop=True)
print(f'Tabla de análisis: {len(df):,} filas')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for ax, col, title in zip(axes[0], ['object_shape', 'object_xpos', 'object_ypos'],
                          ['Forma (código)', 'Posición x (código)', 'Posición y (código)']):
    vc = manifest[col].value_counts().sort_index()
    ax.bar(vc.index.astype(str), vc.values, color=SERIES[0], width=0.6)
    ax.set_title(title); ax.set_ylabel('imágenes'); ax.grid(axis='x', visible=False)
bins = np.linspace(0, 1, 25); centers = (bins[:-1] + bins[1:]) / 2
hue_rgb = [colorsys.hsv_to_rgb(h, .75, .9) for h in centers]     # aquí el color es el dato
for ax, col, title in zip(axes[1], ['object_color', 'spotlight_color', 'background_color'],
                          ['Tono del objeto', 'Tono del foco', 'Tono del fondo']):
    counts, _ = np.histogram(manifest[col], bins=bins)
    ax.bar(centers, counts, width=bins[1] - bins[0], color=hue_rgb, edgecolor=SURF, linewidth=1)
    ax.set_title(title); ax.set_xlabel('hue ∈ [0, 1)'); ax.set_ylabel('imágenes'); ax.grid(axis='x', visible=False)
fig.suptitle('Factores generativos en M3Di-test', x=0.01, ha='left', fontweight='semibold')
fig.tight_layout()
savefig(fig, '01_factores_verdaderos')
plt.show()

## 3. Extractores

### 3.1 Léxicos y colores con nombre

* **Forma:** siete clases con sinónimos equivalentes. Se registra la primera clase mencionada y el conjunto de clases (alucinación de forma).
* **Colores con nombre.** Las referencias (y previsiblemente los modelos ajustados y VL‑JEPA) usan nombres de Matplotlib
  (`"xkcd:bright yellow"`, `"tab:green"`). Cada nombre se resuelve contra `matplotlib.colors` (se prueba el prefijo más largo de hasta 4 palabras),
  se convierte a HSV y se reemplaza por su categoría básica; si $S<0.2$ o $V<0.2$ se considera acromático.
* **Categorías de tono:** $\theta = 360h$; rojo $[345°,15°)$, naranjo $[15°,40°)$, amarillo $[40°,70°)$, verde $[70°,160°)$, cian $[160°,195°)$,
  azul $[195°,255°)$, morado $[255°,285°)$, rosado $[285°,345°)$. Se reporta también acierto con tolerancia de **una categoría adyacente**.
* **Asignación a entidades:** cada mención cromática va al **fondo** si en las 3 palabras siguientes (o, si no, anteriores) hay un sustantivo de fondo,
  al **foco** si hay uno de iluminación, y al **objeto** en otro caso; los colores coordinados heredan la asignación del anterior.
* **Posición:** *left/center/right* y *top/center/bottom*; *center/middle* completa el eje no mencionado; se eliminan giros como *on top of*.

In [ ]:
SHAPE_LEXICON = {
    'teapot': ['teapot', 'tea pot', 'kettle'], 'hare': ['hare', 'rabbit', 'bunny', 'bunnies'],
    'dragon': ['dragon'], 'cow': ['cow', 'bull', 'ox', 'oxen', 'cattle', 'calf'], 'armadillo': ['armadillo'],
    'horse': ['horse', 'pony', 'stallion', 'mare'], 'head': ['head', 'bust', 'face'],
}
SHAPE_CANON_ORDER = list(SHAPE_LEXICON)


def _term_rx(t):
    return r'\b' + re.escape(t).replace(r'\ ', r'[\s\-]?') + r'(?:s|es)?\b'


SHAPE_RE = {k: re.compile('|'.join(_term_rx(t) for t in v), re.I) for k, v in SHAPE_LEXICON.items()}


def find_shapes(text):
    return sorted((m.start(), k) for k, rx in SHAPE_RE.items() for m in rx.finditer(text))


COLOR_CATS = ['red', 'orange', 'yellow', 'green', 'cyan', 'blue', 'purple', 'pink']
HUE_UPPER = [15, 40, 70, 160, 195, 255, 285, 345]
COLOR_LEXICON = {
    'red': ['red', 'reddish', 'crimson', 'scarlet', 'maroon', 'burgundy', 'ruby', 'cherry'],
    'orange': ['orange', 'orangish', 'amber', 'tangerine', 'peach', 'rust', 'copper', 'salmon', 'coral'],
    'yellow': ['yellow', 'yellowish', 'gold', 'golden', 'mustard', 'lemon'],
    'green': ['green', 'greenish', 'lime', 'olive', 'chartreuse', 'emerald', 'mint'],
    'cyan': ['cyan', 'teal', 'turquoise', 'aqua', 'aquamarine'],
    'blue': ['blue', 'bluish', 'navy', 'azure', 'cobalt', 'sapphire', 'cerulean'],
    'purple': ['purple', 'purplish', 'violet', 'lavender', 'indigo', 'lilac', 'plum', 'mauve'],
    'pink': ['pink', 'pinkish', 'magenta', 'fuchsia', 'rose'],
    'achromatic': ['white', 'black', 'gray', 'grey', 'silver', 'brown', 'beige', 'tan'],
}
TERM2CAT = {t: c for c, v in COLOR_LEXICON.items() for t in v}
COLOR_TERM_RE = re.compile(r'\b(' + '|'.join(sorted(map(re.escape, TERM2CAT), key=len, reverse=True)) + r')\b', re.I)
BG_WORDS = {'background', 'backdrop', 'backgrounds', 'wall', 'walls', 'floor', 'ground', 'surface', 'sky',
            'surroundings', 'environment', 'gradient', 'setting', 'canvas', 'plane', 'behind'}
LIGHT_WORDS = {'light', 'lights', 'lighting', 'lit', 'spotlight', 'spotlights', 'glow', 'glowing', 'illumination',
               'illuminated', 'illuminating', 'beam', 'beams', 'shadow', 'shadows', 'highlight', 'highlights',
               'reflection', 'reflections', 'tint', 'shine', 'shining'}
WORD_RE = re.compile(r"[a-z]+(?:'[a-z]+)?")


def hue_to_cat(h):
    deg = (float(h) * 360.0) % 360.0
    if deg >= 345 or deg < 15:
        return 'red'
    return next(c for c, u in zip(COLOR_CATS[1:], HUE_UPPER[1:]) if deg < u)


def rgb_to_cat(rgb):
    h, s, v = colorsys.rgb_to_hsv(*rgb)
    return 'achromatic' if (s < 0.2 or v < 0.2) else hue_to_cat(h)


def cat_distance(a, b):
    if a not in COLOR_CATS or b not in COLOR_CATS:
        return np.inf
    d = abs(COLOR_CATS.index(a) - COLOR_CATS.index(b))
    return min(d, len(COLOR_CATS) - d)


def circ_diff(a, b):
    return (np.asarray(a, float) - np.asarray(b, float) + 0.5) % 1.0 - 0.5


# ── Colores con nombre de Matplotlib ─────────────────────────────────────────
NAMED = {**{k.lower(): v for k, v in mcolors.XKCD_COLORS.items()},
         **{k.lower(): v for k, v in mcolors.TABLEAU_COLORS.items()},
         **{f'css:{k.lower()}': v for k, v in mcolors.CSS4_COLORS.items()}}
PREFIX_RE = re.compile(r'\b(xkcd|tab|css4?)\s*:\s*', re.I)
NAME_WORD_RE = re.compile(r"[A-Za-z][A-Za-z'\-]*")
CAT_WORD = {c: c for c in COLOR_CATS} | {'achromatic': 'gray'}


def find_named_colors(text):
    # Lista de (inicio, fin, nombre, rgb) para cada 'xkcd:…'/'tab:…' reconocido (prefijo más largo).
    out, pos = [], 0
    for m in PREFIX_RE.finditer(text):
        if m.start() < pos:
            continue
        pref = 'css' if m.group(1).lower().startswith('css') else m.group(1).lower()
        words, j = [], m.end()
        for w in NAME_WORD_RE.finditer(text, m.end()):
            if w.start() != j and text[j:w.start()].strip() != '':
                break
            words.append(w); j = w.end()
            if len(words) == 4:
                break
        for k in range(len(words), 0, -1):
            name = f"{pref}:{' '.join(w.group(0) for w in words[:k]).lower()}"
            if name in NAMED:
                out.append((m.start(), words[k - 1].end(), name, mcolors.to_rgb(NAMED[name])))
                pos = words[k - 1].end()
                break
    return out


def replace_named_colors(text):
    hits = find_named_colors(text)
    for s, e, _, rgb in reversed(hits):
        text = text[:s] + CAT_WORD[rgb_to_cat(rgb)] + text[e:]
    return text


def assign_colors(text):
    out = []
    for sent in re.split(r'(?<=[.!?;])\s+|\n+', replace_named_colors(text).lower()):
        prev_end, prev_ent = None, None
        for m in COLOR_TERM_RE.finditer(sent):
            cat = TERM2CAT[m.group(1).lower()]
            gap = sent[prev_end:m.start()] if prev_end is not None else None
            if gap is not None and re.fullmatch(r'\s*(?:,|and|or|to|-|/)?\s*', gap):
                ent = prev_ent
            else:
                after, before = WORD_RE.findall(sent[m.end():])[:3], WORD_RE.findall(sent[:m.start()])[-3:]
                ent = ('background' if any(w in BG_WORDS for w in after) else
                       'spotlight' if any(w in LIGHT_WORDS for w in after) else
                       'background' if any(w in BG_WORDS for w in before) else
                       'spotlight' if any(w in LIGHT_WORDS for w in before) else 'object')
            out.append((cat, ent))
            prev_end, prev_ent = m.end(), ent
    return out


POS_STRIP = re.compile(r'\bon top of\b|\bright[\s\-]angled?\b|\ball right\b|\bright now\b|\bleftover\b|\bleft over\b', re.I)


def parse_position(text):
    t = POS_STRIP.sub(' ', text.lower())
    L, R = bool(re.search(r'\bleft\b', t)), bool(re.search(r'\bright\b', t))
    T, B = bool(re.search(r'\b(top|upper)\b', t)), bool(re.search(r'\b(bottom|lower)\b', t))
    C = bool(re.search(r'\b(center|centre|centered|centred|middle|central)\b', t))
    h = 'ambiguous' if (L and R) else 'left' if L else 'right' if R else None
    v = 'ambiguous' if (T and B) else 'top' if T else 'bottom' if B else None
    if C:
        h, v = h or 'center', v or 'center'
    return h, v

### 3.2 Calibración con la referencia

Los códigos enteros no traen nombre: la biyección código → etiqueta se infiere del texto de referencia, y la **pureza**
$\max_k \hat P(\text{etiqueta}=k\mid\text{código})$ mide la precisión del extractor sobre un texto cuyo contenido es conocido.
Una pureza baja en algún atributo invalida la lectura de ese atributo en el resto del notebook.

In [ ]:
refs = df.drop_duplicates('_key')[['_key', 'image_id', 'reference'] + ATTR_COLS].reset_index(drop=True)
refs['ref_shape'] = refs['reference'].map(lambda t: (find_shapes(t) or [(None, None)])[0][1])
refs[['ref_h', 'ref_v']] = refs['reference'].map(parse_position).apply(pd.Series)


def code_map(codes, labels, name):
    ct = pd.crosstab(codes, labels.fillna('∅'))
    valid = ct.drop(columns=[c for c in ('∅', 'ambiguous') if c in ct.columns])
    mapping = valid.idxmax(axis=1)
    purity = (valid.max(axis=1) / ct.sum(axis=1)).rename('pureza')
    print(f'{name}: pureza ponderada = {valid.max(axis=1).sum() / ct.values.sum():.4f}')
    if mapping.duplicated().any():
        print(f'[aviso] {name}: la asignación no es biyectiva → {mapping.to_dict()}')
    display(pd.concat([ct, purity], axis=1).style.format({'pureza': '{:.3f}'}).set_caption(name))
    return mapping.to_dict()


SHAPE_NAMES = code_map(refs['object_shape'], refs['ref_shape'], 'Forma')
XPOS_NAMES = code_map(refs['object_xpos'], refs['ref_h'], 'Posición x')
YPOS_NAMES = code_map(refs['object_ypos'], refs['ref_v'], 'Posición y')
for c in sorted(manifest['object_shape'].unique()):
    SHAPE_NAMES.setdefault(c, SHAPE_CANON_ORDER[c] if c < len(SHAPE_CANON_ORDER) else str(c))
print('SHAPE_NAMES =', SHAPE_NAMES, '\nXPOS_NAMES  =', XPOS_NAMES, '\nYPOS_NAMES  =', YPOS_NAMES)

### 3.3 El factor `object_color` como tono y la discretización

Con el color nombrado en la referencia se obtiene $h^{\text{ref}}$ y se compara con el factor mediante
$\Delta = \big((h^{\text{ref}}-h^{\text{obj}}+\tfrac12)\bmod 1\big)-\tfrac12$. La concordancia entre la categoría del nombre y la categoría de $h^{\text{obj}}$
es una **cota superior** de lo que un modelo que imite perfectamente la referencia puede lograr en la métrica exacta; por eso se reporta también ±1.

In [ ]:
def ref_color(text):
    hits = find_named_colors(text)
    if not hits:
        return np.nan, None, None
    _, _, name, rgb = hits[0]
    return colorsys.rgb_to_hsv(*rgb)[0], name, rgb_to_cat(rgb)


refs[['ref_hue', 'ref_color_name', 'ref_color_cat']] = refs['reference'].map(ref_color).apply(pd.Series)
ok = refs['ref_hue'].notna()
dlt = 360 * circ_diff(refs.loc[ok, 'ref_hue'], refs.loc[ok, 'object_color'])
cat_obj = refs.loc[ok, 'object_color'].map(hue_to_cat)
UPPER_EXACT = (refs.loc[ok, 'ref_color_cat'] == cat_obj).mean()
UPPER_ADJ = np.mean([cat_distance(a, b) <= 1 for a, b in zip(refs.loc[ok, 'ref_color_cat'], cat_obj)])
print(f'Referencias con color nombrado reconocido: {ok.mean():.1%}  ·  nombres distintos: {refs["ref_color_name"].nunique()}')
print(f'|Δ| mediana = {np.median(np.abs(dlt)):.1f}°, P90 = {np.percentile(np.abs(dlt), 90):.1f}°')
print(f'Cota de concordancia categoría(nombre) vs categoría(h_obj): exacta {UPPER_EXACT:.1%} · ±1 {UPPER_ADJ:.1%}')

fig, axes = plt.subplots(1, 2, figsize=(11, 3.9))
axes[0].scatter(refs.loc[ok, 'object_color'], refs.loc[ok, 'ref_hue'], s=6, alpha=0.35, color=SERIES[0], linewidths=0)
axes[0].plot([0, 1], [0, 1], color=INK3, lw=1, ls='--')
axes[0].set(xlabel='$h^{obj}$ (factor)', ylabel='$h^{ref}$ (nombre en la referencia)', xlim=(0, 1), ylim=(0, 1),
            title='Tono del factor vs. tono del nombre')
axes[1].hist(dlt, bins=np.arange(-180, 181, 5), color=SERIES[0])
axes[1].axvline(0, color=INK3, lw=1, ls='--')
axes[1].set(xlabel='Δ circular (grados)', ylabel='imágenes', title='Distribución de Δ'); axes[1].grid(axis='x', visible=False)
fig.tight_layout()
savefig(fig, '02_validacion_tono_referencia')
plt.show()

### 3.4 Extracción sobre las predicciones

Para cada atributo $a$: `a_pred`, `a_true`, `a_mentioned` $=\mathbb 1[\hat z_a\neq\varnothing]$ y `a_correct` $=\mathbb 1[\hat z_a=z_a]$
(una omisión cuenta como error). Para el color del objeto se agrega el acierto contra la **categoría del nombre en la referencia** (`color_obj_ref`),
que separa "describe mal la imagen" de "no usa la misma convención que la referencia".

In [ ]:
STOP = set('a an the of and or in on at to is are was were be been being it its this that with as by for from into onto over under '
           'which while there their they them image picture shows show showing depicts depicting features featuring appears '
           'appear seems seem has have having can could would will also very some any one two three more most than then '
           'render rendered rendering 3d scene view photo object colored coloured color colour'.split())


def content_tokens(t):
    t = re.sub(r'\b(?:xkcd|tab|css4?)\s*:', ' ', t.lower())
    return [w for w in WORD_RE.findall(t) if w not in STOP]


def unigram_f1(pred, ref):
    p, r = Counter(content_tokens(pred)), Counter(content_tokens(ref))
    ov = sum((p & r).values())
    if not ov:
        return 0.0
    pr, rc = ov / sum(p.values()), ov / sum(r.values())
    return 2 * pr * rc / (pr + rc)


def extract_row(pred, ref):
    shapes = find_shapes(pred)
    h, v = parse_position(pred)
    cols = assign_colors(pred)
    first = lambda ent: next((c for c, e in cols if e == ent), None)
    words = WORD_RE.findall(pred.lower())
    tri = list(zip(words, words[1:], words[2:]))
    return {'shape_pred': shapes[0][1] if shapes else None, 'shape_set': tuple(sorted({k for _, k in shapes})),
            'xpos_pred': h, 'ypos_pred': v, 'color_obj_pred': first('object'), 'color_bg_pred': first('background'),
            'color_spot_pred': first('spotlight'), 'n_color_mentions': len(cols),
            'n_words': len(words), 'n_chars': len(pred),
            'n_sents': len([s for s in re.split(r'(?<=[.!?])\s+', pred.strip()) if s]),
            'rep_trigram': 1 - len(set(tri)) / len(tri) if tri else 0.0,
            'ends_clean': bool(re.search(r'[.!?)"\'\]}]\s*$', pred.strip())),
            'uses_named_color': bool(PREFIX_RE.search(pred)), 'unigram_f1': unigram_f1(pred, ref)}


t = time.time()
ext = pd.DataFrame([extract_row(p, r) for p, r in zip(df['prediction'], df['reference'])], index=df.index)
df = pd.concat([df, ext], axis=1)
print(f'Extracción: {len(df):,} filas en {time.time() - t:.0f} s')

df['shape_true'] = df['object_shape'].map(SHAPE_NAMES)
df['xpos_true'] = df['object_xpos'].map(XPOS_NAMES)
df['ypos_true'] = df['object_ypos'].map(YPOS_NAMES)
df['color_obj_true'] = df['object_color'].map(hue_to_cat)
df['color_bg_true'] = df['background_color'].map(hue_to_cat)
df['color_spot_true'] = df['spotlight_color'].map(hue_to_cat)
df = df.merge(refs[['_key', 'ref_color_cat']], on='_key', how='left')

ATTRS = ['shape', 'xpos', 'ypos', 'color_obj', 'color_bg', 'color_spot']
for a in ATTRS:
    df[f'{a}_mentioned'] = df[f'{a}_pred'].notna()
    df[f'{a}_correct'] = df[f'{a}_pred'].eq(df[f'{a}_true']) & df[f'{a}_mentioned']
for a in ['color_obj', 'color_bg', 'color_spot']:
    df[f'{a}_adj_mentioned'] = df[f'{a}_mentioned']
    df[f'{a}_adj_correct'] = [cat_distance(p, q) <= 1 for p, q in zip(df[f'{a}_pred'], df[f'{a}_true'])]
df['color_obj_ref_mentioned'] = df['color_obj_mentioned']
df['color_obj_ref_correct'] = df['color_obj_pred'].eq(df['ref_color_cat']) & df['color_obj_mentioned']
df['pos_joint_mentioned'] = df['xpos_mentioned'] & df['ypos_mentioned']
df['pos_joint_correct'] = df['xpos_correct'] & df['ypos_correct']
df['core_mentioned'] = df['shape_mentioned'] & df['pos_joint_mentioned'] & df['color_obj_mentioned']
df['core_correct'] = df['shape_correct'] & df['pos_joint_correct'] & df['color_obj_adj_correct']
df['macro'] = df[['shape_correct', 'pos_joint_correct', 'color_obj_adj_correct']].mean(axis=1)
df['shape_halluc'] = [any(s != q for s in S) for S, q in zip(df['shape_set'], df['shape_true'])]
df['ref_n_words'] = df['reference'].map(lambda s: len(WORD_RE.findall(s.lower())))

ATTR_LABELS = {'shape': 'Forma', 'xpos': 'Posición x', 'ypos': 'Posición y', 'pos_joint': 'Posición (x,y)',
               'color_obj': 'Color objeto', 'color_obj_adj': 'Color objeto ±1', 'color_obj_ref': 'Color obj. = ref.',
               'color_bg': 'Color fondo', 'color_bg_adj': 'Color fondo ±1', 'color_spot': 'Color foco',
               'color_spot_adj': 'Color foco ±1', 'core': 'Forma+pos+color±1'}
df[['run', 'prediction', 'shape_pred', 'shape_true', 'xpos_pred', 'xpos_true', 'ypos_pred', 'ypos_true',
    'color_obj_pred', 'color_obj_true', 'color_bg_pred', 'color_bg_true']].sample(6, random_state=SEED)

## 4. Texto: longitud, truncamiento y vocabulario

* **Truncamiento probable:** el texto no termina en puntuación de cierre (típico de alcanzar `max_new_tokens`).
* **Repetición:** $\rho = 1 - |\{\text{trigramas distintos}\}| / |\{\text{trigramas}\}|$; valores altos indican bucles degenerados.
* **distinct‑$n$** sobre el corpus de la corrida: bajo = salidas plantilla (esperable en prompts restringidos, ajuste fino y recuperación).
* **Uso de colores con nombre** (`xkcd:`/`tab:`): indica si el modelo adoptó la convención de la referencia.

In [ ]:
def q(p):
    f = lambda s: s.quantile(p); f.__name__ = f'p{int(100 * p)}'; return f


def distinct_n(texts, n):
    g = [x for t in texts for x in zip(*[WORD_RE.findall(t.lower())[i:] for i in range(n)])]
    return len(set(g)) / len(g) if g else np.nan


text_tab = (df.groupby('run', sort=False)
              .agg(palabras_media=('n_words', 'mean'), palabras_p50=('n_words', q(.5)), palabras_p95=('n_words', q(.95)),
                   oraciones=('n_sents', 'mean'), vacias=('n_words', lambda s: (s == 0).mean()),
                   truncadas=('ends_clean', lambda s: 1 - s.mean()), repeticion=('rep_trigram', 'mean'),
                   color_con_nombre=('uses_named_color', 'mean'), f1_unigrama=('unigram_f1', 'mean'),
                   unicas=('prediction', lambda s: s.nunique() / len(s)))
              .reindex(RUN_ORDER))
text_tab['distinct_2'] = [distinct_n(df.loc[df['run'] == r, 'prediction'], 2) for r in RUN_ORDER]
pct_cols = ['vacias', 'truncadas', 'color_con_nombre', 'unicas']
display(text_tab.style.format({c: '{:.1%}' for c in pct_cols} |
                              {c: '{:.2f}' for c in text_tab.columns if c not in pct_cols}))
save_table(text_tab, 'texto_por_corrida')
REF_MED = refs['reference'].map(lambda s: len(WORD_RE.findall(s.lower()))).median()
print(f'Referencia: mediana {REF_MED:.0f} palabras')

In [ ]:
h = 0.34 * len(RUN_ORDER) + 1.6
fig, axes = plt.subplots(1, 3, figsize=(14, h), sharey=True, gridspec_kw={'width_ratios': [2.3, 1, 1]})
y = np.arange(len(RUN_ORDER))
ax = axes[0]
for yi, r in zip(y, RUN_ORDER):
    s = df.loc[df['run'] == r, 'n_words'].clip(lower=1)
    q05, q25, q50, q75, q95 = s.quantile([.05, .25, .5, .75, .95])
    c = RUN_COLORS[r]
    ax.plot([q05, q95], [yi, yi], color=c, lw=1.2, alpha=0.6, solid_capstyle='round')
    ax.plot([q25, q75], [yi, yi], color=c, lw=6, alpha=0.85, solid_capstyle='butt')
    ax.plot(q50, yi, '|', color=INK, ms=11, mew=2)
ax.axvline(REF_MED, color=INK2, ls='--', lw=1)
ax.text(REF_MED, len(RUN_ORDER) - 0.35, ' mediana referencia', color=INK2, fontsize=8, va='bottom')
ax.set_xscale('log'); plain_log_x(ax)
ax.set(xlabel='palabras por descripción (escala log; barra = P25–P75, línea = P5–P95, | = mediana)',
       title='Longitud de las predicciones')
ax.set_yticks(y, RUN_ORDER); ax.invert_yaxis(); ax.grid(axis='y', visible=False)

for ax, col, title in zip(axes[1:], ['truncadas', 'color_con_nombre'], ['Truncadas', 'Usa colores xkcd:/tab:']):
    ax.barh(y, text_tab[col].values, height=0.6, color=[RUN_COLORS[r] for r in RUN_ORDER])
    ax.xaxis.set_major_formatter(PCT); ax.set(title=title); ax.grid(axis='y', visible=False)
    ax.set_xlim(0, max(0.05, text_tab[col].max() * 1.1))
fig.tight_layout()
savefig(fig, '03_longitud_truncamiento')
plt.show()

In [ ]:
LEX_WORDS = {w for v in SHAPE_LEXICON.values() for w in v} | set(TERM2CAT) | BG_WORDS | LIGHT_WORDS | \
            {'left', 'right', 'top', 'bottom', 'center', 'centre', 'middle', 'upper', 'lower', 'positioned', 'located',
             'placed', 'side', 'corner', 'dark', 'bright', 'small', 'large', 'figure', 'model', 'sculpture', 'light'}
und = df[~df['shape_mentioned']]
print(f'Predicciones sin ninguna forma del léxico: {len(und):,} ({len(und) / len(df):.1%})')
alias = {}
for s in [SHAPE_NAMES[k] for k in sorted(SHAPE_NAMES)]:
    c = Counter(w for t in und.loc[und['shape_true'] == s, 'prediction'] for w in content_tokens(t) if w not in LEX_WORDS)
    top = [f'{w} ({n})' for w, n in c.most_common(10)]
    alias[s] = top + [''] * (10 - len(top))
alias_df = pd.DataFrame(alias, index=range(1, 11))
display(alias_df.style.set_caption('Términos más frecuentes cuando no se detecta forma, por forma verdadera'))
save_table(alias_df, 'terminos_forma_no_detectada', index=False)

## 5. Fidelidad por atributo

Para la corrida $r$ y el atributo $a$ ($N_r$ imágenes):
$$
\hat c_{r,a}=\tfrac1{N_r}\textstyle\sum_i\mathbb 1[\hat z_{i,a}\neq\varnothing],\qquad
\hat\alpha_{r,a}=\tfrac1{N_r}\textstyle\sum_i\mathbb 1[\hat z_{i,a}=z_{i,a}],\qquad
\hat\alpha^{\mid}_{r,a}=\hat\alpha_{r,a}/\hat c_{r,a}.
$$
$\hat\alpha$ penaliza omisión y error; $\hat\alpha^{\mid}$ aísla la calidad de lo que sí se dice. Intervalos de **Wilson** al $1-\alpha$:
$\tilde p\pm\frac{z}{1+z^2/n}\sqrt{\frac{\hat p(1-\hat p)}{n}+\frac{z^2}{4n^2}}$, con $\tilde p=\frac{\hat p+z^2/(2n)}{1+z^2/n}$.
La **línea base mayoritaria** $\max_k\hat\pi_k$ es la exactitud de responder siempre la clase más frecuente.
El **índice macro** por imagen es $\tfrac13(\mathbb 1_{\text{forma}}+\mathbb 1_{\text{pos}(x,y)}+\mathbb 1_{\text{color}\pm1})$.

In [ ]:
def wilson(k, n, alpha=ALPHA):
    if n == 0:
        return np.nan, np.nan
    z = stats.norm.ppf(1 - alpha / 2); p = k / n; den = 1 + z ** 2 / n
    c = (p + z ** 2 / (2 * n)) / den; hw = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / den
    return c - hw, c + hw


METRIC_ATTRS = ['shape', 'xpos', 'ypos', 'pos_joint', 'color_obj', 'color_obj_adj', 'color_obj_ref',
                'color_bg', 'color_bg_adj', 'color_spot', 'color_spot_adj', 'core']
meta_idx = runs_meta.set_index('run')
rows = []
for r, g in df.groupby('run', sort=False):
    n = len(g)
    for a in METRIC_ATTRS:
        k, m = int(g[f'{a}_correct'].sum()), int(g[f'{a}_mentioned'].sum())
        lo, hi = wilson(k, n)
        rows.append(dict(run=r, exp=meta_idx.at[r, 'exp'], model=meta_idx.at[r, 'model_label'],
                         prompt=meta_idx.at[r, 'prompt_label'], attr=a, n=n, mencion=m / n, exactitud=k / n,
                         ic_lo=lo, ic_hi=hi, exactitud_cond=k / m if m else np.nan))
acc = pd.DataFrame(rows)
macro = df.groupby('run')['macro'].mean()

base = df.drop_duplicates('_key')
MAJ = {a: base[f'{a}_true'].value_counts(normalize=True).iloc[0] for a in ATTRS}
MAJ['pos_joint'] = (base['xpos_true'] + base['ypos_true']).value_counts(normalize=True).iloc[0]
MAJ['color_obj_ref'] = base['ref_color_cat'].value_counts(normalize=True).iloc[0]
for a in ['color_obj', 'color_bg', 'color_spot']:
    vc = base[f'{a}_true'].value_counts(normalize=True)
    MAJ[f'{a}_adj'] = max(sum(vc.get(c2, 0) for c2 in COLOR_CATS if cat_distance(c1, c2) <= 1) for c1 in COLOR_CATS)

acc_wide = acc.pivot(index='run', columns='attr', values='exactitud').reindex(index=RUN_ORDER, columns=METRIC_ATTRS)
acc_wide['macro'] = macro.reindex(RUN_ORDER)
acc_wide.columns = [ATTR_LABELS.get(c, 'Macro') for c in acc_wide.columns]
display(acc_wide.style.format('{:.3f}').background_gradient(cmap=SEQ, vmin=0, vmax=1)
        .set_caption('Exactitud (omisión = error)'))
save_table(acc, 'fidelidad_por_atributo_largo', index=False)
save_table(acc_wide, 'fidelidad_por_atributo')
print('Línea base mayoritaria:', {ATTR_LABELS[k]: round(float(v), 3) for k, v in MAJ.items()})

# Mejor prompt por (etapa, modelo) según el índice macro: base de las secciones 6–8
best = (runs_meta.assign(macro=runs_meta['run'].map(macro))
        .sort_values('macro', ascending=False).groupby(['exp', 'model_label'], sort=False).head(1)
        .sort_values(['exp', 'model_label'], key=lambda s: s.map(
            {**{e: i for i, e in enumerate(EXP_ORDER)}, **{m: i for i, m in enumerate(MODEL_ORDER)}})))
BEST_RUNS = best['run'].tolist()
display(best[['exp', 'model_label', 'prompt_label', 'macro']].rename(columns={'macro': 'índice macro'})
        .style.format({'índice macro': '{:.3f}'}).hide(axis='index').set_caption('Mejor prompt por modelo'))

**Figura 4 — Mención, exactitud y exactitud condicional.** Tres heatmaps alineados por corrida. Una celda clara en *mención* pero oscura en
*condicional* indica un atributo que el modelo omite pero acierta cuando lo nombra (problema de formato/prompt, no de percepción).

In [ ]:
sub = ['shape', 'xpos', 'ypos', 'color_obj', 'color_bg', 'color_spot']
fig, axes = plt.subplots(1, 3, figsize=(15, 0.36 * len(RUN_ORDER) + 1.9), sharey=True)
for ax, metric, title in zip(axes, ['mencion', 'exactitud', 'exactitud_cond'],
                             ['Tasa de mención $\\hat c$', 'Exactitud $\\hat\\alpha$', 'Exactitud condicional $\\hat\\alpha^{\\mid}$']):
    M = acc.pivot(index='run', columns='attr', values=metric).reindex(index=RUN_ORDER, columns=sub).values
    heat(ax, M, [ATTR_LABELS[a] for a in sub], RUN_ORDER, fontsize=7)
    ax.set_title(title); ax.tick_params(axis='x', rotation=35)
    for lab in ax.get_xticklabels():
        lab.set_ha('right')
fig.tight_layout()
savefig(fig, '04_heatmap_mencion_exactitud')
plt.show()

**Figura 5 — Exactitud con IC de Wilson.** Pequeños múltiplos por atributo; la línea punteada es la base mayoritaria.
Para el color se muestran tres criterios: categoría exacta, ±1 categoría y coincidencia con la categoría del nombre en la referencia.

In [ ]:
plot_attrs = ['shape', 'pos_joint', 'color_obj', 'color_obj_adj', 'color_obj_ref', 'color_bg_adj', 'core']
fig, axes = plt.subplots(1, len(plot_attrs), figsize=(2.25 * len(plot_attrs) + 2.6, 0.34 * len(RUN_ORDER) + 1.8), sharey=True)
y = np.arange(len(RUN_ORDER))
for ax, a in zip(axes, plot_attrs):
    s = acc[acc['attr'] == a].set_index('run').reindex(RUN_ORDER)
    for yi, r in zip(y, RUN_ORDER):
        ax.plot([s.at[r, 'ic_lo'], s.at[r, 'ic_hi']], [yi, yi], color=RUN_COLORS[r], lw=2, solid_capstyle='round')
        ax.plot(s.at[r, 'exactitud'], yi, 'o', ms=6, color=RUN_COLORS[r], mec=SURF, mew=1.2)
    if a in MAJ:
        ax.axvline(MAJ[a], color=INK3, ls=':', lw=1.2)
    ax.set_xlim(0, 1); ax.xaxis.set_major_formatter(PCT); ax.set_xticks([0, .5, 1])
    ax.set_title(ATTR_LABELS[a], fontsize=9.5); ax.grid(axis='y', visible=False)
axes[0].set_yticks(y, RUN_ORDER); axes[0].invert_yaxis()
handles = [plt.Line2D([], [], marker='o', ls='', color=MODEL_COLORS[m], label=m) for m in MODEL_ORDER] + \
          [plt.Line2D([], [], color=INK3, ls=':', label='base mayoritaria')]
fig.tight_layout(rect=(0, 0, 1, 0.93))
fig.legend(handles=handles, loc='upper center', ncol=len(handles), bbox_to_anchor=(0.5, 0.985))
savefig(fig, '05_exactitud_ic_wilson')
plt.show()

**Figura 6 — Descomposición del resultado.** Para cada corrida, la proporción de imágenes donde el atributo es *correcto*, *incorrecto*
(mencionado pero errado) u *omitido*. Distingue modelos que se equivocan de modelos que simplemente no lo dicen.

In [ ]:
OUT_COLS = {'correcto': SERIES[0], 'incorrecto': SERIES[1], 'omitido': NEUTRAL}
dec_attrs = [('shape', 'Forma'), ('pos_joint', 'Posición (x,y)'), ('color_obj_adj', 'Color objeto ±1'), ('color_bg_adj', 'Color fondo ±1')]
fig, axes = plt.subplots(1, len(dec_attrs), figsize=(3.2 * len(dec_attrs) + 2.6, 0.34 * len(RUN_ORDER) + 1.8), sharey=True)
for ax, (a, title) in zip(axes, dec_attrs):
    g = df.groupby('run', sort=False)
    corr = g[f'{a}_correct'].mean().reindex(RUN_ORDER).values
    ment = g[f'{a}_mentioned'].mean().reindex(RUN_ORDER).values
    parts = {'correcto': corr, 'incorrecto': ment - corr, 'omitido': 1 - ment}
    left = np.zeros(len(RUN_ORDER))
    for k, v in parts.items():
        ax.barh(y, v, left=left, height=0.7, color=OUT_COLS[k], edgecolor=SURF, linewidth=1.5, label=k)
        left += v
    ax.set_xlim(0, 1); ax.xaxis.set_major_formatter(PCT); ax.set_title(title); ax.grid(axis='y', visible=False)
axes[0].set_yticks(y, RUN_ORDER); axes[0].invert_yaxis()
hdl, lbl = axes[0].get_legend_handles_labels()
fig.tight_layout(rect=(0, 0, 1, 1 - 0.5 / (0.34 * len(RUN_ORDER) + 1.8)))
fig.legend(hdl, lbl, loc='upper center', ncol=3, bbox_to_anchor=(0.5, 1.0))
savefig(fig, '06_descomposicion_resultado')
plt.show()

## 6. Patrones entre corridas

**Figura 7 — Sensibilidad al prompt.** Exactitud por prompt, una línea por modelo (línea continua = primera etapa, discontinua = siguientes).
Una pendiente pronunciada indica que el resultado depende más de *cómo se pregunta* que de *lo que el modelo percibe*.

In [ ]:
if len(PROMPT_ORDER) > 1:
    sens_attrs = ['shape', 'pos_joint', 'color_obj_adj', 'macro']
    fig, axes = plt.subplots(1, len(sens_attrs), figsize=(3.6 * len(sens_attrs) + 1, 3.8), sharey=True)
    x = np.arange(len(PROMPT_ORDER))
    for ax, a in zip(axes, sens_attrs):
        for (e, m), g in runs_meta.groupby(['exp', 'model_label'], sort=False):
            vals = [macro[r] if a == 'macro' else acc.set_index(['run', 'attr']).at[(r, a), 'exactitud'] for r in g['run']]
            xs = [PROMPT_ORDER.index(p) for p in g['prompt_label']]
            order = np.argsort(xs)
            ax.plot(np.array(xs)[order], np.array(vals)[order], marker=EXP_MARKERS[e], color=MODEL_COLORS[m],
                    ls='-' if e == EXP_ORDER[0] else '--', ms=6, mec=SURF, mew=1)
        if a in MAJ:
            ax.axhline(MAJ[a], color=INK3, ls=':', lw=1.1)
        ax.set_xticks(x, PROMPT_ORDER, rotation=30, ha='right'); ax.set_ylim(0, 1); ax.yaxis.set_major_formatter(PCT)
        ax.set_title(ATTR_LABELS.get(a, 'Índice macro'))
    handles = [plt.Line2D([], [], color=MODEL_COLORS[m], marker='o', label=m) for m in MODEL_ORDER]
    if MULTI_EXP:
        handles += [plt.Line2D([], [], color=INK2, marker=EXP_MARKERS[e], ls='-' if i == 0 else '--', label=e)
                    for i, e in enumerate(EXP_ORDER)]
    fig.tight_layout(rect=(0, 0, 1, 0.9))
    fig.legend(handles=handles, loc='upper center', ncol=len(handles), bbox_to_anchor=(0.5, 0.99))
    savefig(fig, '07_sensibilidad_prompt')
    plt.show()

**Figura 8 — Cobertura vs. exactitud condicional.** Cada punto es una corrida; la diagonal punteada es $\hat\alpha^{\mid}=\hat c$.
Arriba a la izquierda: modelos que dicen poco pero bien; abajo a la derecha: modelos que lo dicen todo pero se equivocan.
Las isolíneas grises son exactitud estricta constante $\hat\alpha=\hat c\,\hat\alpha^{\mid}$.

In [ ]:
cov_attrs = ['shape', 'pos_joint', 'color_obj_adj', 'color_bg_adj']
fig, axes = plt.subplots(1, len(cov_attrs), figsize=(3.5 * len(cov_attrs) + 0.6, 3.9), sharey=True)
cc = np.linspace(0.01, 1, 200)
for ax, a in zip(axes, cov_attrs):
    for lvl in (0.25, 0.5, 0.75):
        ax.plot(cc[cc >= lvl], lvl / cc[cc >= lvl], color=GRID, lw=1.2, zorder=0)
        ax.text(lvl + 0.015, 0.985, f'α={lvl:.2f}', fontsize=7, color=INK3, ha='left', va='top')
    s = acc[acc['attr'] == a]
    for _, rr in s.iterrows():
        ret = meta_idx.at[rr['run'], 'retrieval']
        ax.scatter(rr['mencion'], rr['exactitud_cond'], s=46, marker=EXP_MARKERS[rr['exp']],
                   facecolor='none' if ret else MODEL_COLORS[rr['model']], edgecolor=MODEL_COLORS[rr['model']],
                   linewidth=1.5, zorder=3)
    ax.set(xlim=(-0.02, 1.02), ylim=(-0.02, 1.02), xlabel='cobertura $\\hat c$', title=ATTR_LABELS[a])
    ax.xaxis.set_major_formatter(PCT); ax.yaxis.set_major_formatter(PCT)
axes[0].set_ylabel('exactitud condicional $\\hat\\alpha^{\\mid}$')
handles = [plt.Line2D([], [], marker='o', ls='', color=MODEL_COLORS[m], label=m) for m in MODEL_ORDER]
if runs_meta['retrieval'].any():
    handles.append(plt.Line2D([], [], marker='o', ls='', mfc='none', mec=INK2, label='recuperación (hueco)'))
if MULTI_EXP:
    handles += [plt.Line2D([], [], marker=EXP_MARKERS[e], ls='', color=INK2, label=e) for e in EXP_ORDER]
fig.tight_layout(rect=(0, 0, 1, 0.88))
fig.legend(handles=handles, loc='upper center', ncol=len(handles), bbox_to_anchor=(0.5, 0.99))
savefig(fig, '08_cobertura_vs_condicional')
plt.show()

**Figura 9 — Similitud textual no es fidelidad.** Izquierda: F1 de unigramas contra la referencia vs. índice macro de atributos
(se reporta la correlación de Spearman $\rho_s$ entre corridas). Derecha: longitud media vs. índice macro.
Si las métricas de superficie fueran buenas proxies de fidelidad, los puntos se ordenarían sobre una diagonal.

In [ ]:
run_lvl = pd.DataFrame({'f1': df.groupby('run')['unigram_f1'].mean(), 'words': df.groupby('run')['n_words'].mean(),
                        'macro': macro}).reindex(RUN_ORDER).join(meta_idx[['exp', 'model_label', 'retrieval']])
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2), sharey=True)
for ax, xcol, xlab in zip(axes, ['f1', 'words'], ['F1 unigrama vs. referencia (media)', 'palabras por descripción (media, log)']):
    for r, rr in run_lvl.iterrows():
        c = MODEL_COLORS[rr['model_label']]
        ax.scatter(rr[xcol], rr['macro'], s=55, marker=EXP_MARKERS[rr['exp']], facecolor='none' if rr['retrieval'] else c,
                   edgecolor=c, linewidth=1.5, zorder=3)
    rho, p = stats.spearmanr(run_lvl[xcol], run_lvl['macro'])
    ax.text(0.02, 0.97, f'$\\rho_s$ = {rho:+.2f} (p = {p:.2g}, n = {len(run_lvl)})', transform=ax.transAxes,
            va='top', fontsize=8.5, color=INK2)
    ax.set(xlabel=xlab); ax.yaxis.set_major_formatter(PCT)
axes[1].set_xscale('log'); plain_log_x(axes[1])
axes[0].set_ylabel('índice macro de atributos')
fig.tight_layout(rect=(0, 0, 1, 0.9))
fig.legend(handles=handles, loc='upper center', ncol=len(handles), bbox_to_anchor=(0.5, 0.99))
savefig(fig, '09_similitud_vs_fidelidad')
plt.show()

**Figura 10 — Comparación entre etapas** (sólo si hay más de una). Para cada modelo presente en ambas etapas se compara su mejor prompt;
la barra horizontal une ambas exactitudes. Responde a cuánto del error de la etapa inicial era desalineamiento de formato y cuánto falta de percepción.

In [ ]:
if MULTI_EXP:
    st_attrs = ['shape', 'pos_joint', 'color_obj_adj', 'color_obj_ref', 'macro']
    both = best.groupby('model_label').filter(lambda g: g['exp'].nunique() > 1)
    mods = [m for m in MODEL_ORDER if m in set(both['model_label'])]
    if mods:
        fig, axes = plt.subplots(1, len(st_attrs), figsize=(2.9 * len(st_attrs) + 1.5, 0.55 * len(mods) + 1.6), sharey=True)
        for ax, a in zip(axes, st_attrs):
            for yi, m in enumerate(mods):
                vals = []
                for e in EXP_ORDER:
                    rr = both[(both['model_label'] == m) & (both['exp'] == e)]
                    if len(rr):
                        r = rr['run'].iat[0]
                        vals.append((e, macro[r] if a == 'macro' else acc.set_index(['run', 'attr']).at[(r, a), 'exactitud']))
                ax.plot([v for _, v in vals], [yi] * len(vals), color=GRID, lw=4, solid_capstyle='round', zorder=1)
                for e, v in vals:
                    ax.scatter(v, yi, s=60, marker=EXP_MARKERS[e], color=MODEL_COLORS[m], edgecolor=SURF, zorder=3)
            ax.set_xlim(0, 1); ax.xaxis.set_major_formatter(PCT); ax.set_title(ATTR_LABELS.get(a, 'Índice macro'), fontsize=9.5)
            ax.grid(axis='y', visible=False)
        axes[0].set_yticks(range(len(mods)), mods); axes[0].invert_yaxis()
        eh = [plt.Line2D([], [], marker=EXP_MARKERS[e], ls='', color=INK2, label=f'etapa {e}') for e in EXP_ORDER]
        fig.tight_layout(rect=(0, 0, 1, 0.86))
        fig.legend(handles=eh, loc='upper center', ncol=len(eh), bbox_to_anchor=(0.5, 0.99))
        savefig(fig, '10_comparacion_etapas')
        plt.show()
    else:
        print('Ningún modelo aparece en más de una etapa.')
else:
    print('Una sola etapa: sección omitida.')

## 7. Análisis de errores (mejor prompt de cada modelo)

Para no multiplicar paneles con 23 corridas, las figuras de esta sección usan la corrida de mejor índice macro por *(etapa, modelo)*.

**Figura 11 — Confusión de forma** (filas normalizadas, $\hat P(\hat s=k\mid s=j)$; **∅** = sin forma del léxico).

In [ ]:
SHAPE_ORDER = [SHAPE_NAMES[k] for k in sorted(SHAPE_NAMES)]
nb_ = len(BEST_RUNS); ncol = min(4, nb_); nrow = int(np.ceil(nb_ / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4.1 * ncol, 3.8 * nrow), squeeze=False)
for ax, r in zip(axes.flat, BEST_RUNS):
    g = df[df['run'] == r]
    ct = pd.crosstab(g['shape_true'], g['shape_pred'].fillna('∅')).reindex(index=SHAPE_ORDER, columns=SHAPE_ORDER + ['∅'], fill_value=0)
    heat(ax, (ct.T / ct.sum(axis=1).replace(0, np.nan)).T.values, SHAPE_ORDER + ['∅'], SHAPE_ORDER, fontsize=6.5)
    ax.set_title(r, fontsize=9); ax.tick_params(axis='x', rotation=45, labelsize=7.5); ax.tick_params(axis='y', labelsize=7.5)
    for lab in ax.get_xticklabels():
        lab.set_ha('right')
for ax in axes.flat[nb_:]:
    ax.axis('off')
fig.supxlabel('forma predicha'); fig.supylabel('forma verdadera')
fig.tight_layout()
savefig(fig, '11_confusion_forma')
plt.show()

**Figura 12 — Posición en la grilla $3\times3$.** Exactitud conjunta $(\hat x,\hat y)$ por celda verdadera. Revela sesgos espaciales
(p. ej. describir todo como *center*) y asimetrías entre ejes. La tabla siguiente muestra la distribución marginal de lo que predice cada corrida.

In [ ]:
X_ORDER = [XPOS_NAMES[k] for k in sorted(XPOS_NAMES)]
Y_ORDER = [YPOS_NAMES[k] for k in sorted(YPOS_NAMES)]
fig, axes = plt.subplots(nrow, ncol, figsize=(3.3 * ncol, 3.1 * nrow), squeeze=False)
for ax, r in zip(axes.flat, BEST_RUNS):
    g = df[df['run'] == r]
    M = g.pivot_table(index='ypos_true', columns='xpos_true', values='pos_joint_correct', aggfunc='mean').reindex(index=Y_ORDER, columns=X_ORDER)
    heat(ax, M.values, X_ORDER, Y_ORDER)
    ax.set_title(r, fontsize=9)
for ax in axes.flat[nb_:]:
    ax.axis('off')
fig.suptitle('Exactitud de posición conjunta por celda verdadera', x=0.01, ha='left', fontweight='semibold')
fig.tight_layout()
savefig(fig, '12_posicion_grilla')
plt.show()

pos_bias = pd.concat({a: df[df['run'].isin(BEST_RUNS)].groupby('run')[f'{a}_pred']
                      .value_counts(normalize=True, dropna=False).unstack(fill_value=0).reindex(BEST_RUNS)
                      for a in ['xpos', 'ypos']}, axis=1)
pos_bias.columns = [f'{a}={v if isinstance(v, str) else "∅"}' for a, v in pos_bias.columns]
display(pos_bias.style.format('{:.2f}').background_gradient(cmap=SEQ, vmin=0, vmax=1).set_caption('Distribución de lo predicho'))

**Figura 13 — Color del objeto a lo largo del círculo cromático.** Exactitud (±1 categoría) en ventanas de $15°$ de $h^{\text{obj}}$;
la banda inferior muestra el tono de cada ventana y la línea gris la concordancia de la propia referencia (cota del generador de texto).
Caídas localizadas señalan regiones que el modelo nombra de forma inconsistente (típicamente las transiciones amarillo–verde y azul–morado).

In [ ]:
hb = np.linspace(0, 1, 25); hc = (hb[:-1] + hb[1:]) / 2
df['hue_bin'] = np.clip(np.digitize(df['object_color'], hb) - 1, 0, 23)
refs['hue_bin'] = np.clip(np.digitize(refs['object_color'], hb) - 1, 0, 23)
ref_adj = pd.Series([cat_distance(a, hue_to_cat(b)) <= 1 for a, b in zip(refs['ref_color_cat'], refs['object_color'])],
                    index=refs.index).groupby(refs['hue_bin']).mean().reindex(range(24))
fig, axes = plt.subplots(1, len(EXP_ORDER), figsize=(6.2 * len(EXP_ORDER), 3.9), sharey=True, squeeze=False)
for ax, e in zip(axes[0], EXP_ORDER):
    ax.plot(hc * 360, ref_adj.values, color=INK3, lw=1.5, ls='--', label='referencia')
    for r in best.loc[best['exp'] == e, 'run']:
        s = df[df['run'] == r].groupby('hue_bin')['color_obj_adj_correct'].mean().reindex(range(24))
        ax.plot(hc * 360, s.values, '-o', ms=3.5, color=RUN_COLORS[r], label=meta_idx.at[r, 'model_label'])
    for i in range(24):
        ax.axvspan(hb[i] * 360, hb[i + 1] * 360, ymin=0, ymax=0.035, color=colorsys.hsv_to_rgb(hc[i], .75, .9), lw=0)
    ax.set(xlim=(0, 360), ylim=(-0.04, 1.02), xlabel='tono del objeto (grados)', title=f'Etapa {e}' if MULTI_EXP else None)
    ax.set_xticks(range(0, 361, 60)); ax.yaxis.set_major_formatter(PCT)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=5, fontsize=7.5)
axes[0, 0].set_ylabel('exactitud color objeto (±1)')
fig.tight_layout()
savefig(fig, '13_color_vs_tono')
plt.show()

**Figura 14 — Confusión de categorías de color del objeto** (filas normalizadas; incluye *achromatic* y **∅**).

In [ ]:
CC = COLOR_CATS + ['achromatic', '∅']
fig, axes = plt.subplots(nrow, ncol, figsize=(4.3 * ncol, 4.0 * nrow), squeeze=False)
for ax, r in zip(axes.flat, BEST_RUNS):
    g = df[df['run'] == r]
    ct = pd.crosstab(g['color_obj_true'], g['color_obj_pred'].fillna('∅')).reindex(index=COLOR_CATS, columns=CC, fill_value=0)
    heat(ax, (ct.T / ct.sum(axis=1).replace(0, np.nan)).T.values, CC, COLOR_CATS, fontsize=6.3)
    ax.set_title(r, fontsize=9); ax.tick_params(axis='x', rotation=45, labelsize=7.5); ax.tick_params(axis='y', labelsize=7.5)
    for lab in ax.get_xticklabels():
        lab.set_ha('right')
for ax in axes.flat[nb_:]:
    ax.axis('off')
fig.supxlabel('color predicho (objeto)'); fig.supylabel('categoría verdadera')
fig.tight_layout()
savefig(fig, '14_confusion_color_objeto')
plt.show()

**Figura 15 — Contraste objeto–fondo.** Exactitud en función de $|\Delta h| = |h^{\text{obj}}\ominus h^{\text{bg}}|\in[0°,180°]$.
Hipótesis: con objeto y fondo de tono similar, el modelo confunde sus colores y reconoce peor la forma. La tabla reporta, para cada corrida,
$\hat\beta_1$ de $\operatorname{logit}P(\text{acierto})=\beta_0+\beta_1|\Delta h|_{\text{rad}}$ y su valor‑$p$ de Wald.

In [ ]:
df['contrast_deg'] = np.abs(360 * circ_diff(df['object_color'], df['background_color']))
cb = np.arange(0, 181, 15); cm = (cb[:-1] + cb[1:]) / 2
df['contrast_bin'] = np.clip(np.digitize(df['contrast_deg'], cb) - 1, 0, len(cm) - 1)


def logit_1d(x, y, iters=50):
    X = np.column_stack([np.ones_like(x), x]); b = np.zeros(2)
    for _ in range(iters):
        p = 1 / (1 + np.exp(-X @ b)); W = p * (1 - p)
        H = X.T @ (X * W[:, None]); g = X.T @ (y - p)
        try:
            step = np.linalg.solve(H, g)
        except np.linalg.LinAlgError:
            return np.nan, np.nan
        b += step
        if np.abs(step).max() < 1e-8:
            break
    try:
        se = np.sqrt(np.diag(np.linalg.inv(H)))
    except np.linalg.LinAlgError:
        return b[1], np.nan
    return b[1], 2 * stats.norm.sf(abs(b[1] / se[1]))


fig, axes = plt.subplots(1, 3, figsize=(14, 3.9), sharey=True)
for ax, target, title in zip(axes, ['color_obj_adj_correct', 'color_bg_adj_correct', 'shape_correct'],
                             ['Color objeto ±1', 'Color fondo ±1', 'Forma']):
    for r in BEST_RUNS:
        if df.loc[df['run'] == r, target.replace('_correct', '_mentioned')].mean() < 0.05:
            continue  # la corrida casi nunca menciona el atributo: la curva sería ≈0 y no informa
        s = df[df['run'] == r].groupby('contrast_bin')[target].mean().reindex(range(len(cm)))
        ax.plot(cm, s.values, marker=EXP_MARKERS[meta_idx.at[r, 'exp']], ms=4, color=RUN_COLORS[r],
                ls='-' if meta_idx.at[r, 'exp'] == EXP_ORDER[0] else '--', label=r)
    ax.set(xlabel='|Δ tono| objeto–fondo (grados)', title=title, xlim=(0, 180)); ax.set_xticks(range(0, 181, 30))
    ax.yaxis.set_major_formatter(PCT)
axes[0].set_ylabel('exactitud')
axes[-1].legend(loc='center left', bbox_to_anchor=(1.01, 0.5), fontsize=7.5)
fig.tight_layout()
savefig(fig, '15_contraste_objeto_fondo')
plt.show()

lr = []
for r in RUN_ORDER:
    g = df[df['run'] == r]
    for target in ['color_obj_adj_correct', 'shape_correct']:
        b1, p = logit_1d(np.deg2rad(g['contrast_deg'].values), g[target].astype(float).values)
        lr.append(dict(run=r, atributo=target.replace('_correct', ''), beta1=b1, p=p))
lr_tab = pd.DataFrame(lr).pivot(index='run', columns='atributo', values=['beta1', 'p']).reindex(RUN_ORDER)
lr_tab.columns = [f'{b}_{a}' for a, b in lr_tab.columns]
display(lr_tab.style.format('{:.3g}'))
save_table(lr_tab, 'contraste_logit')

## 8. Comparaciones pareadas

Todas las corridas describen **las mismas imágenes**, así que las diferencias se evalúan de forma pareada. Para corridas $A,B$ sea
$b=\#\{A\text{ acierta},B\text{ falla}\}$ y $c=\#\{A\text{ falla},B\text{ acierta}\}$; bajo $H_0$, $b\mid(b+c)\sim\operatorname{Bin}(b+c,\tfrac12)$
(**McNemar exacto**). Los valores‑$p$ se ajustan por **Holm–Bonferroni** dentro de cada familia:
(i) prompts dentro de cada *(etapa, modelo)*; (ii) modelos dentro de cada etapa usando su mejor prompt.

> Elegir el mejor prompt con los mismos datos con que luego se comparan modelos introduce un sesgo optimista (selección *post hoc*);
> las comparaciones (ii) deben leerse como descriptivas o repetirse en una partición independiente.

In [ ]:
def holm(p):
    p = np.asarray(p, float); order = np.argsort(p); m = len(p); adj = np.empty(m); run_ = 0.0
    for k, i in enumerate(order):
        run_ = max(run_, (m - k) * p[i]); adj[i] = min(1.0, run_)
    return adj


TEST_ATTRS = ['shape', 'pos_joint', 'color_obj_adj', 'core']
W = {a: df.pivot_table(index='_key', columns='run', values=f'{a}_correct', aggfunc='first') for a in TEST_ATTRS}


def mcnemar_rows(pairs, family):
    out = []
    for A, B in pairs:
        for a in TEST_ATTRS:
            P = W[a][[A, B]].dropna().astype(bool)
            b = int((P[A] & ~P[B]).sum()); c = int((~P[A] & P[B]).sum())
            out.append(dict(familia=family, A=A, B=B, atributo=ATTR_LABELS[a], n=len(P), acc_A=P[A].mean(), acc_B=P[B].mean(),
                            delta=P[A].mean() - P[B].mean(), b=b, c=c,
                            p=stats.binomtest(b, b + c, 0.5).pvalue if b + c else 1.0))
    return out


rows = []
for (e, m), g in runs_meta.groupby(['exp', 'model_label'], sort=False):
    rows += mcnemar_rows(list(itertools.combinations(g['run'], 2)), f'prompts · {e} · {m}')
for e, g in best.groupby('exp', sort=False):
    rows += mcnemar_rows(list(itertools.combinations(g['run'], 2)), f'modelos · {e}')
mc = pd.DataFrame(rows)
if len(mc):
    mc['p_holm'] = mc.groupby('familia')['p'].transform(lambda s: holm(s.values))
    mc['signif'] = mc['p_holm'] < ALPHA
    save_table(mc, 'mcnemar', index=False)
    fmt = {'acc_A': '{:.3f}', 'acc_B': '{:.3f}', 'delta': '{:+.3f}', 'p': '{:.1e}', 'p_holm': '{:.1e}'}
    display(mc[mc['familia'].str.startswith('modelos')].style.format(fmt)
            .apply(lambda r: ['font-weight: bold' if r['signif'] else ''] * len(r), axis=1).hide(axis='index'))
    print(f"Comparaciones de prompts: {mc['familia'].str.startswith('prompts').sum()} "
          f"({mc.loc[mc['familia'].str.startswith('prompts'), 'signif'].mean():.0%} significativas tras Holm) → ver mcnemar.csv")

**Figura 16 — Δ exactitud entre modelos** (fila − columna, mejor prompt de cada uno; \* = significativo tras Holm; azul = la fila es mejor).

In [ ]:
mcm = mc[mc['familia'].str.startswith('modelos')] if len(mc) else mc
if len(mcm):
    show = ['Forma', 'Posición (x,y)', 'Color objeto ±1']
    fams = list(dict.fromkeys(mcm['familia']))
    lim = max(0.05, mcm['delta'].abs().max())
    fig, axes = plt.subplots(len(fams), len(show), figsize=(3.8 * len(show) + 1, 3.4 * len(fams)), squeeze=False)
    for i, fam in enumerate(fams):
        runs_f = best.loc[best['exp'] == fam.split(' · ')[1], 'run'].tolist()
        labs = [meta_idx.at[r, 'model_label'] for r in runs_f]
        for j, a in enumerate(show):
            s = mcm[(mcm['familia'] == fam) & (mcm['atributo'] == a)]
            M = pd.DataFrame(np.nan, index=runs_f, columns=runs_f); S = M.isna() & False
            for _, rr in s.iterrows():
                M.at[rr['A'], rr['B']], M.at[rr['B'], rr['A']] = rr['delta'], -rr['delta']
                S.at[rr['A'], rr['B']] = S.at[rr['B'], rr['A']] = rr['signif']
            ax = axes[i, j]
            heat(ax, M.values, labs, labs, cmap=DIV, vmin=-lim, vmax=lim, fmt='{:+.2f}')
            for t_, (rc, v) in zip(ax.texts, [(rc, v) for rc, v in np.ndenumerate(M.values) if not np.isnan(v)]):
                if S.values[rc]:
                    t_.set_text(t_.get_text() + '*')
            ax.set_title(f'{a} · {fam.split(" · ")[1]}' if MULTI_EXP else a, fontsize=9.5)
            ax.tick_params(axis='x', rotation=30, labelsize=7.5); ax.tick_params(axis='y', labelsize=7.5)
            for lab in ax.get_xticklabels():
                lab.set_ha('right')
    fig.tight_layout()
    savefig(fig, '16_mcnemar_modelos')
    plt.show()

## 9. Dificultad por imagen y ejemplos

La dificultad de la imagen $i$ en un atributo, sobre las $R$ corridas de la **primera etapa**, es $d_i=1-\frac1R\sum_r\mathbb 1[\hat z_{i,r}=z_i]$.
Con errores independientes entre corridas, $d_i$ se concentraría en torno a su media; una masa en $d_i=1$ (todas fallan) indica configuraciones
de factores intrínsecamente difíciles. La tabla cruza la dificultad con la forma y con el contraste objeto–fondo.

In [ ]:
e0 = df[df['exp'] == EXP_ORDER[0]]
R = e0['run'].nunique()
diff = (e0.groupby('_key')[['shape_correct', 'pos_joint_correct', 'color_obj_adj_correct']].mean().rsub(1)
          .join(e0.drop_duplicates('_key').set_index('_key')[['image_id', 'shape_true', 'contrast_deg']]))
fig, axes = plt.subplots(1, 3, figsize=(12, 3.3), sharey=True)
edges = np.linspace(-0.5 / R, 1 + 0.5 / R, R + 2)
for ax, c, tt in zip(axes, ['shape_correct', 'pos_joint_correct', 'color_obj_adj_correct'], ['Forma', 'Posición (x,y)', 'Color objeto ±1']):
    ax.hist(diff[c], bins=edges, color=SERIES[0], edgecolor=SURF, linewidth=1)
    ax.set(title=tt, xlabel=f'fracción de las {R} corridas que fallan'); ax.xaxis.set_major_formatter(PCT)
    ax.grid(axis='x', visible=False)
axes[0].set_ylabel('imágenes')
fig.tight_layout()
savefig(fig, '17_dificultad_por_imagen')
plt.show()

diff['contraste'] = pd.cut(diff['contrast_deg'], [0, 30, 60, 120, 180], include_lowest=True).astype(str)
tabs = [diff.groupby('shape_true')[['shape_correct', 'pos_joint_correct', 'color_obj_adj_correct']].mean(),
        diff.groupby('contraste')[['shape_correct', 'pos_joint_correct', 'color_obj_adj_correct']].mean()]
for tb, cap in zip(tabs, ['Dificultad media por forma verdadera', 'Dificultad media por |Δ tono| objeto–fondo']):
    tb.columns = ['Forma', 'Posición (x,y)', 'Color objeto ±1']
    display(tb.style.format('{:.3f}').background_gradient(cmap=SEQ, vmin=0, vmax=1).set_caption(cap))
save_table(tabs[0], 'dificultad_por_forma')

In [ ]:
SHOW = ['run', 'image_id', 'prediction', 'reference', 'shape_true', 'shape_pred', 'xpos_true', 'xpos_pred',
        'ypos_true', 'ypos_pred', 'color_obj_true', 'color_obj_pred', 'color_bg_true', 'color_bg_pred']


def show_examples(frame, n=N_EXAMPLES, title=None):
    if title:
        display(Markdown(f'**{title}**'))
    if frame.empty:
        print('(sin casos)'); return
    display(frame.sample(min(n, len(frame)), random_state=SEED)[SHOW].reset_index(drop=True))


for r in BEST_RUNS:
    show_examples(df[df['run'] == r], title=f'Muestra aleatoria · {r}')
show_examples(df[df['n_words'] > 5].nlargest(300, 'rep_trigram'), 4, 'Mayor repetición (posibles bucles)')
show_examples(df[~df['ends_clean'] & (df['n_words'] > 0)], 4, 'Probablemente truncadas')
hard = diff.index[(diff[['shape_correct', 'color_obj_adj_correct']] == 1).all(axis=1)]
show_examples(df[df['_key'].isin(hard) & df['run'].isin(BEST_RUNS)], 6, 'Imágenes que ninguna corrida de la 1.ª etapa describe bien')

## 10. Exportación

In [ ]:
keep = (['exp_id', 'exp', 'model', 'model_label', 'prompt', 'run', 'retrieval', 'image_id', 'prediction', 'reference'] + ATTR_COLS +
        [c for a in ATTRS for c in (f'{a}_true', f'{a}_pred', f'{a}_mentioned', f'{a}_correct')] +
        ['ref_color_cat', 'color_obj_adj_correct', 'color_obj_ref_correct', 'color_bg_adj_correct', 'pos_joint_correct',
         'core_correct', 'macro', 'shape_halluc', 'n_words', 'rep_trigram', 'ends_clean', 'uses_named_color', 'unigram_f1',
         'contrast_deg'])
out = df[keep].copy()
out['retrieval'] = out['retrieval'].astype(bool)
pl.from_dict({c: out[c].tolist() for c in out.columns}, strict=False).write_parquet(TAB_DIR / 'eda_predicciones_atributos.parquet')

summary = (acc_wide[['Forma', 'Posición (x,y)', 'Color objeto', 'Color objeto ±1', 'Color obj. = ref.', 'Forma+pos+color±1', 'Macro']]
           .join(text_tab[['palabras_media', 'truncadas', 'color_con_nombre', 'distinct_2']]))
display(summary.style.format('{:.3f}').background_gradient(cmap=SEQ, vmin=0, vmax=1, subset=summary.columns[:7]))
save_table(summary, 'resumen_eda')
print(f'Tiempo total: {(time.time() - T0) / 60:.1f} min')
print('Figuras:', *sorted(p.name for p in FIG_DIR.glob('*.png')), sep='\n  ')

## Notas metodológicas y limitaciones

1. **Extracción léxica.** Las tasas dependen de los léxicos de §3; la pureza de §3.2 acota su error sobre texto con formato de referencia, pero
   las descripciones libres de la etapa sin ajuste son más variadas. Revise la tabla de §4 (términos sin forma detectada) y amplíe `SHAPE_LEXICON`
   sólo con sinónimos legítimos (*dragon → dinosaur* es un error, no un sinónimo).
2. **Color.** La asignación objeto/fondo/foco es una heurística de ventana. La discretización del tono es arbitraria en los bordes; §3.3 cuantifica
   cuánto discrepa el propio generador de M3Di de ella (cota superior de la métrica exacta). `Color obj. = ref.` mide adopción de la convención de la referencia.
3. **Posición.** *left/right/top* referidos a la luz u otras partes producen falsos positivos; `ambiguous` (ambos extremos de un eje) cuenta como error.
4. **VL‑JEPA** recupera captions del *pool*: su similitud textual (Fig. 9) no es comparable con la de los generativos; sí lo es su fidelidad.
5. **Inferencia.** Las corridas están pareadas por imagen: usar McNemar, no comparar IC de Wilson solapados. El mejor prompt se elige con los mismos datos
   (§8), por lo que las comparaciones entre modelos son descriptivas.